In [142]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle
from pykeen.triples import TriplesFactory
from pykeen.models import DistMult, RotatE, TransE, TransD
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from pykeen.optimizers import Adam
from pykeen.regularizers import LpRegularizer
from sklearn.decomposition import PCA, TruncatedSVD
import glob
from tqdm import tqdm

In [115]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [121]:
files = glob.glob("data/data_from_neo4j/*.csv")
dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

main_data = pd.concat(dfs, ignore_index=True)
main_data = main_data[main_data["id_1"] != main_data["id_2"]]
main_data = main_data.astype(str)

In [122]:
#main_data = pd.read_csv('data/PPI_1M.csv')
#main_data = main_data.astype(str)

triples = main_data[['id_1', 'interaction', 'id_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)


In [123]:
with open("data/dict.pkl", "rb") as f:
    emb_dict = pickle.load(f)

In [138]:
entity2id = list(triplet_data.entity_to_id.keys())

In [171]:
EMB = torch.stack([emb_dict[int(i)] for i in entity2id], dim=0)

# pca = PCA(n_components=64)
# X_pca = pca.fit_transform(EMB.numpy())
# EMB = torch.from_numpy(X_pca)

In [172]:
X = EMB.float()
dataset = TensorDataset(X)
loader = DataLoader(dataset, batch_size=1024, shuffle=True)

D_in = X.shape[1]
D_latent = 128  # целевая размерность

class AE(nn.Module):
    def __init__(self, d_in, d_latent):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, d_in),
        )

    def forward(self, x):
        z = self.encoder(x).squeeze()
        x_hat = self.decoder(z).squeeze()
        return x_hat, z

ae = AE(D_in, D_latent).to(device)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

ae.train()
for epoch in range(30):
    total = 0.0
    for (xb,) in loader:
        xb = xb.to(device)
        x_hat, _ = ae(xb)
        loss = loss_fn(x_hat, xb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1}: loss={total/len(dataset):.6f}")

# Получить сжатые эмбеддинги [N, D_latent]
ae.eval()
with torch.no_grad():
    Z = ae.encoder(X.to(device)).cpu()
EMB = Z

epoch 1: loss=0.019146
epoch 2: loss=0.005757
epoch 3: loss=0.003781
epoch 4: loss=0.003169
epoch 5: loss=0.002852
epoch 6: loss=0.002635
epoch 7: loss=0.002471
epoch 8: loss=0.002371
epoch 9: loss=0.002264
epoch 10: loss=0.002182
epoch 11: loss=0.002114
epoch 12: loss=0.002059
epoch 13: loss=0.002011
epoch 14: loss=0.001963
epoch 15: loss=0.001935
epoch 16: loss=0.001895
epoch 17: loss=0.001866
epoch 18: loss=0.001837
epoch 19: loss=0.001813
epoch 20: loss=0.001800
epoch 21: loss=0.001769
epoch 22: loss=0.001750
epoch 23: loss=0.001734
epoch 24: loss=0.001718
epoch 25: loss=0.001707
epoch 26: loss=0.001699
epoch 27: loss=0.001681
epoch 28: loss=0.001667
epoch 29: loss=0.001668
epoch 30: loss=0.001644


In [173]:
EMB_DIM = EMB.shape[1]
LR = 1e-3
MARGIN = 1.1
WEIGHT = 1e-3
EPOCHS = 1
BATCH_SIZE = 4096
NUM_NEGS_PER_POS = 10

loss_function = MarginRankingLoss(margin=MARGIN)

model = TransE(
    triples_factory=training_set,
    embedding_dim=EMB_DIM,
    random_seed=100,
    loss = loss_function,
    regularizer=LpRegularizer,
    regularizer_kwargs=dict(p=2, weight=WEIGHT),

)
model = model.to(device)

#загрузка эмбеддингов
with torch.no_grad():
    model.entity_representations[0]._embeddings.weight.copy_(EMB)

model.entity_representations[0]._embeddings.weight.requires_grad_(False)

optimizer = Adam(params=model.parameters(), lr=LR)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='pseudotyped',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

# training_loop.train(
#     num_epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     triples_factory=training_set,
#     use_tqdm_batch=False,
# )

evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples.to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],
)


metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Evaluating on cuda:0: 100%|██████████| 196k/196k [05:12<00:00, 628triple/s] 


,Side,Rank_type,Metric,Value
5,both,realistic,variance,3.075431e+08
14,both,realistic,adjusted_arithmetic_mean_rank,7.531728e-01
23,both,realistic,standard_deviation,1.753691e+04
32,both,realistic,inverse_median_rank,5.308137e-05
41,both,realistic,median_rank,1.883900e+04
50,both,realistic,arithmetic_mean_rank,2.224985e+04
59,both,realistic,z_geometric_mean_rank,2.978078e+02
68,both,realistic,geometric_mean_rank,1.141636e+04
77,both,realistic,inverse_geometric_mean_rank,8.759357e-05
86,both,realistic,z_inverse_harmonic_mean_rank,4.438606e+02
